## Streaming
<p>Este notebookimplementa um fluxo definido em duas etapas</p>
<ol>
<li>Arquivos *.csv para camada Bronze</li>
<li>Partindo da camada Bronze, realiza a agregação definida até a gold</li>
</ol>

In [0]:
# Imports
from pathlib import Path
from pyspark.sql import functions as F

##### Definição das constantes

In [0]:
CATALOG    = "ANP_Combustiveis"

# Schemas & volume
rawSchema    = "00_raw"
bronzeSchema = "01_bronze"
goldSchema   = "03_gold"
volume       = "data"

# Definição de caminhos
volumePath = Path(f"/Volumes/{CATALOG}/{rawSchema}/{volume}")
paths = {
    "PRECOS_CSV":        Path(volumePath / "ANP" / "PRECOS" / "CSV"),
    "CHECKPOINT_BRONZE": Path(volumePath / "STREAMING" / "CHECKPOINT_BRONZE"),
    "CHECKPOINT_GOLD":   Path(volumePath / "STREAMING" / "CHECKPOINT_GOLD"),
}

# Definição de tabelas
TABLE_BRONZE = (f"`{CATALOG}`.`{bronzeSchema}`.`precos_revenda_fluxo`")
TABLE_GOLD   = (f"`{CATALOG}`.`{goldSchema}`.`precos_semanais_fluxo`")

##### Leitura dos arquivos .csv
<p>Leitura dos arquivos com Spark</p>

In [0]:
# Definição do schema dos dados
pricesSchema = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(str(paths["PRECOS_CSV"] / "*.csv"))
    .schema
)

# Leitura dos dados
streamPricesRawDf = (
    spark.readStream
    .schema(pricesSchema)
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .option("pathGlobFilter", "*.csv")
    .option("maxFilesPerTrigger", 1)
    .csv(str(paths["PRECOS_CSV"]))
)

##### Preparação da camada Bronze

In [0]:
# Deinifição do schema da tabela bronze
streamPricesBronzeDf = (
    streamPricesRawDf
    # Seleção apenas das colunas necessárias
    .select(
        F.upper(F.trim(F.col("Regiao - Sigla"))).alias("regiao"),
        F.upper(F.trim(F.col("Estado - Sigla"))).alias("uf"),
        F.trim(F.col("Municipio")).alias("municipio"),
        F.trim(F.col("Revenda")).alias("razao_social"),
        F.trim(F.col("CNPJ da Revenda")).alias("cnpj_revenda"),
        F.upper(F.trim(F.col("Produto"))).alias("produto"),
        F.to_date(F.col("Data da Coleta"), "dd/MM/yyyy").alias("data_coleta"),
        F.to_timestamp(F.col("Data da Coleta"),"dd/MM/yyyy").alias("tempo_evento"),
        F.regexp_replace(F.col("Valor de Venda"), ",", ".").cast("decimal(10,3)")
         .alias("valor_venda"),
        F.upper(F.trim(F.col("Unidade de Medida"))).alias("unidade_medida"),
        F.col("_metadata.file_name").alias("arquivo_origem"),
        F.current_timestamp().alias("data_hora_ingestao"),
    )
    # Filtro para dados nulos
    .filter(
        F.col("tempo_evento").isNotNull()
        & F.col("valor_venda").isNotNull()
        & (F.col("valor_venda") > 0)
    )
)

'''
Gravação na tabela Bronze
- checkpointLocation: local onde será gravado o checkpoint da stream
- trigger: tempo de processamento da stream
- queryName: nome da query que será gravada no checkpoint
- toTable: nome da tabela que será gravada
'''
queryBronze = (
    streamPricesBronzeDf.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        str(paths["CHECKPOINT_BRONZE"]),
    )
    .trigger(availableNow=True)
    .queryName("precos_revenda_bronze_fluxo")
    .toTable(TABLE_BRONZE)
)

##### Watermark e agregação da tabela Bronze

In [0]:
# Leitura da tabela bronze
streamPricesDf_weekly = (
    spark.readStream
    .table(TABLE_BRONZE)
    .withWatermark("tempo_evento", "30 days")
    .groupBy(
        F.window("tempo_evento", "7 days"),
        "uf",
        "municipio",
        "produto",
    )
    .agg(
        F.round(F.avg("valor_venda"), 3)
        .cast("decimal(10,3)")
        .alias("preco_medio"),
        F.min("valor_venda").alias("preco_minimo"),
        F.max("valor_venda").alias("preco_maximo"),
        F.count("*").alias("quantidade_observacoes"),
    )
    .select(
        F.col("window.start").alias("inicio_janela"),
        F.col("window.end").alias("fim_janela"),
        "uf",
        "municipio",
        "produto",
        "preco_medio",
        "preco_minimo",
        "preco_maximo",
        "quantidade_observacoes",
        F.current_timestamp().alias("processado_gold_em"),
    )
)

##### Gravação na tabela Gold

In [0]:
queryGold = (
    streamPricesDf_weekly.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        str(paths["CHECKPOINT_GOLD"]),
    )
    .trigger(availableNow=True)
    .queryName("precos_semanais_gold_fluxo")
    .toTable(TABLE_GOLD)
)